# Teste de diferentes classificadores para Isquemia

In [2]:
import pandas as pd
import numpy as np
import joblib 
import os
import cv2
import time
from sklearn.model_selection import KFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from ocpc_py import MultiClassPC
from codecarbon import EmissionsTracker

# Configurações
base_path = "../dataset/ischaemia/"
image_size = (64, 64)

# Monta dataset
dataset = []
for classe, label in zip(["Aug-Positive", "Aug-Negative"], [1, 0]):
    pasta = os.path.join(base_path, classe)
    for imagem in os.listdir(pasta):
        caminho_imagem = os.path.join(pasta, imagem)
        dataset.append((caminho_imagem, label))

df = pd.DataFrame(dataset, columns=["imagem", "label"])

# Carrega e normaliza imagens
def load_images(df, image_size):
    imagens, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row["imagem"])
        if img is not None:
            img = cv2.resize(img, image_size)
            img = img.astype("float32") / 255.0
            imagens.append(img.flatten())
            labels.append(row["label"])
        else:
            print(f"Imagem não carregada: {row['imagem']}")
    return np.array(imagens), np.array(labels)

X_all, y_all = load_images(df, image_size)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

print("\n[Validação Cruzada - Treinamento]")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    pca = PCA(n_components=50)
    X_tr_pca = pca.fit_transform(X_tr)
    X_val_pca = pca.transform(X_val)

    clf = MultiClassPC()
    clf.fit(X_tr_pca, y_tr)
    y_pred = clf.predict(X_val_pca)
    y_proba = clf.predict_proba(X_val_pca)[:, 1]

    accs.append(accuracy_score(y_val, y_pred))
    precs.append(precision_score(y_val, y_pred))
    recs.append(recall_score(y_val, y_pred))
    f1s.append(f1_score(y_val, y_pred))
    aucs.append(roc_auc_score(y_val, y_proba))

    print(f"\n[Fold {fold}]")
    print(f"Acurácia: {accs[-1]:.4f} | Precisão: {precs[-1]:.4f} | Recall: {recs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUC: {aucs[-1]:.4f}")

# Resultados finais do K-Fold
print("\n[Métricas Médias - Cross-Validation]")
print(f"Acurácia: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Precisão: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Recall:   {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"AUC:      {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

# TREINAMENTO FINAL PARA PRODUÇÃO
print("\n[Treinamento Final e Avaliação em Teste]")
tracker = EmissionsTracker(log_level="ERROR")  # Desabilitar logs detalhados
tracker.start()

pca_final = PCA(n_components=50)
X_train_pca = pca_final.fit_transform(X_train)
X_test_pca = pca_final.transform(X_test)

clf_final = MultiClassPC()
clf_final.fit(X_train_pca, y_train)

emissions = tracker.stop()
print(f"\n[Pegada de Carbono]")
print(f"Emissões estimadas durante todo o experimento: {emissions:.6f} kg CO₂eq")

# Avaliação no conjunto de teste
y_pred_test = clf_final.predict(X_test_pca)
y_proba_test = clf_final.predict_proba(X_test_pca)[:, 1]

print("\n[Desempenho no Conjunto de Teste]")
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_proba_test):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_test))

output_dir = "../models/ischaemia/"
os.makedirs(output_dir, exist_ok=True)
pca_path = os.path.join(output_dir, "OCPC_pca_ischaemia.pkl")
clf_path = os.path.join(output_dir, "OCPC_modelo_ischaemia.pkl")
joblib.dump(pca_final, pca_path)
joblib.dump(clf_final, clf_path)


[Validação Cruzada - Treinamento]

[Fold 1]
Acurácia: 0.7209 | Precisão: 0.7427 | Recall: 0.6739 | F1: 0.7066 | AUC: 0.7869

[Fold 2]
Acurácia: 0.7023 | Precisão: 0.7382 | Recall: 0.6548 | F1: 0.6940 | AUC: 0.7746

[Fold 3]
Acurácia: 0.7125 | Precisão: 0.7243 | Recall: 0.6761 | F1: 0.6993 | AUC: 0.7838

[Fold 4]
Acurácia: 0.7163 | Precisão: 0.7315 | Recall: 0.6654 | F1: 0.6969 | AUC: 0.7887


[codecarbon WARNING @ 19:22:03] Multiple instances of codecarbon are allowed to run at the same time.



[Fold 5]
Acurácia: 0.6979 | Precisão: 0.7218 | Recall: 0.6460 | F1: 0.6818 | AUC: 0.7879

[Métricas Médias - Cross-Validation]
Acurácia: 0.7100 ± 0.0086
Precisão: 0.7317 ± 0.0080
Recall:   0.6632 ± 0.0114
F1-Score: 0.6957 ± 0.0081
AUC:      0.7844 ± 0.0052

[Treinamento Final e Avaliação em Teste]

[Pegada de Carbono]
Emissões estimadas durante todo o experimento: 0.000021 kg CO₂eq

[Desempenho no Conjunto de Teste]
Acurácia: 0.7057
Precisão: 0.7207
Recall:   0.6717
F1-Score: 0.6953
AUC:      0.7662
Matriz de Confusão:
[[730 257]
 [324 663]]


['../models/ischaemia/OCPC_modelo_ischaemia.pkl']